# Altay Tank, Motor için multilingual embedding modeliyle BERTopic analizi

In [1]:
import pandas as pd

In [2]:
!pip install bertopic sentence-transformers umap-learn hdbscan pandas

In [4]:
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

In [10]:
altay_tank_haber = "Altay_Motor_temiz.csv"

df = pd.read_csv(altay_tank_haber, sep=";", encoding="utf-8-sig")
docs = df["metin_temiz"].tolist()
print(f"Toplam doküman: {len(docs)}")

Toplam doküman: 31


## Embedding

In [11]:
print("Embedding modeli indiriliyor ...")
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
embeddings = embedding_model.encode(docs, show_progress_bar=True)
print(f"Embedding matric boyutu : {embeddings.shape}")

Embedding modeli indiriliyor ...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding matric boyutu : (31, 384)


### Türkçe Stopword -TF-IDF

In [ ]:
TURKCE_STOPWORDS = [
    "acaba", "altmış", "altı", "ama", "ancak", "arada", "aslında", "ayrıca",
    "bana", "bazı", "belki", "ben", "benden", "beni", "benim", "beri", "bile",
    "bin", "bir", "birçok", "biri", "birkaç", "birkez", "birşey", "biz",
    "bizden", "bize", "bizi", "bizim", "bu", "buna", "bunda", "bundan", "bunu",
    "bunun", "burada", "böyle", "böylece", "da", "daha", "dahi", "de", "defa",
    "değil", "diğer", "diye", "doksan", "dokuz", "dolayı", "dolayısıyla",
    "dört", "edecek", "eden", "ederek", "edilecek", "ediliyor", "edilmesi",
    "ediyor", "eğer", "elli", "en", "etmesi", "etti", "ettiği", "ettiğini",
    "gibi", "göre", "halen", "hangi", "hatta", "hem", "henüz", "hep", "hepsi",
    "her", "herhangi", "herkesin", "hiç", "hiçbir", "için", "iki", "ile",
    "ilgili", "ise", "işte", "itibaren", "itibariyle", "kadar", "karşın",
    "kendi", "kendilerine", "kendini", "kendisi", "kendisine", "kendisini",
    "kez", "ki", "kim", "kimden", "kime", "kimi", "kimse", "kırk", "milyar",
    "milyon", "mu", "mü", "mı", "nasıl", "ne", "neden", "nedenle", "nerde",
    "nerede", "nereye", "niye", "niçin", "o", "olan", "olarak", "oldu",
    "olduğu", "olduğunu", "olduklarını", "olmadı", "olmadığı", "olmak",
    "olması", "olmayan", "olmaz", "olsa", "olsun", "olup", "olur", "olursa",
    "oluyor", "on", "ona", "ondan", "onlar", "onlardan", "onları", "onların",
    "onu", "onun", "otuz", "oysa", "öyle", "pek", "rağmen", "sadece", "sanki",
    "sekiz", "seksen", "sen", "senden", "seni", "senin", "siz", "sizden",
    "sizi", "sizin", "sonra", "şey", "şeyden", "şeyi", "şeyler", "şöyle",
    "şu", "şuna", "şunda", "şundan", "şunları", "şunu", "tarafından", "tüm",
    "üç", "üzere", "var", "vardı", "ve", "veya", "ya", "yani", "yapacak",
    "yapılan", "yapılması", "yapıyor", "yapmak", "yaptı", "yaptığı",
    "yaptığını", "yaptıkları", "yedi", "yerine", "yetmiş", "yine", "yirmi",
    "yoksa", "yüz", "zaten",
]
 
vectorizer_model = CountVectorizer(stop_words=TURKCE_STOPWORDS, ngram_range=(1, 2))

### Keşfedici Mod, HDBSCAN kümeleme

In [19]:
print("/n" + "=" * 60)
print("A) Keşfedici Mod")
print("="*60)

umap_model = UMAP(n_neighbors=5, n_components=5, min_dist=0.0, random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=3, min_samples=1, metric="euclidean")


topic_model = BERTopic(
    embedding_model = embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    language="turkish",
    calculate_probabilities=False,
    verbose=True,
)

topics, _ = topic_model.fit_transform(docs, embeddings=embeddings)

print("/n--- Konu Özeti ---")
print(topic_model.get_topic_info()[["Topic", "Count", "Name"]].to_string(index=False))

print("/n--- Her konunun temsilci kelimeleri ---")

for t in sorted(set(topics)):
    if t == -1:
        print("Topic -1 (outliner): hiçbir kümeye net uymayan haberler")
        continue
    kelimeler = ", ".join([w for w, _ in topic_model.get_topic(t)[:8]])
    print(f"Topic {t}: {kelimeler}")

2026-07-01 18:03:56,486 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-07-01 18:03:56,516 - BERTopic - Dimensionality - Completed ✓
2026-07-01 18:03:56,516 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-07-01 18:03:56,518 - BERTopic - Cluster - Completed ✓
2026-07-01 18:03:56,519 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-07-01 18:03:56,526 - BERTopic - Representation - Completed ✓


/n============================================================
A) Keşfedici Mod
/n--- Konu Özeti ---
 Topic  Count                                        Name
     0     17                 0_motor_türkiye_yerli_altay
     1      8                 1_alman_ihracat_motor_altay
     2      3                            2_t1_t2_güç_2025
     3      3 3_katar_savunma_stratejik savunma_stratejik
/n--- Her konunun temsilci kelimeleri ---
Topic 0: motor, türkiye, yerli, altay, bmc, ana, güç, den
Topic 1: alman, ihracat, motor, altay, nedeniyle, muharebe tankı, tankı, seri
Topic 2: t1, t2, güç, 2025, adet, 85, 165, 2028
Topic 3: katar, savunma, stratejik savunma, stratejik, motor, tankın, platformunun, muharebe


In [20]:
if len(set(topics) - {-1}) > 1:
    fig_hier = topic_model.visualize_hierarchy()
    fig_hier.write_html("hiyerarsi.html")
    fig_heat = topic_model.visualize_heatmap()
    fig_heat.write_html("heatmap.html")
    print("/nGörselleştirmeler kaydedildi: hiyerarsi.html, heatmap.html")

/nGörselleştirmeler kaydedildi: hiyerarsi.html, heatmap.html


In [21]:
df["kesif_topic"] = topics
df.to_csv("altay_kesif_sonuc.csv", index=False)

### Zero-shot Mod - kategorilerin önceden verilmesi

In [27]:
print("/n" + "=" * 60)
print("B) Zero-shot Mod")
print("="*60)

zeroshot_kategoriler = [
    "Almanya ambargosu ihracat lisansı kısıtlaması",
    "Güney Kore'den motor tedariki anlaşması", 
    "yerli Batu motor geliştirme ve ikame",
    "tank teslimatı ve genel platform haberleri"
]

zeroshot_model = BERTopic(
    embedding_model = embedding_model,
    vectorizer_model = vectorizer_model,
    zeroshot_topic_list = zeroshot_kategoriler,
    zeroshot_min_similarity=0.35,
    umap_model=umap_model,
    hdbscan_model = hdbscan_model,
    language="turkish",
)

zs_topics, _ = zeroshot_model.fit_transform(docs, embeddings=embeddings)

print("/n--- Zero-shot konu özeti ---")
print(zeroshot_model.get_topic_info()[["Topic", "Count", "Name"]].to_string(index=False))

df["zeroshot_topic"] = zs_topics
df.to_csv("altay_zeroshot_sonuc.csv", index=False)

print("Tamamlandı. Çıktı dosyaları: altay_kesif_sonuc.csv, altay_zeroshot_sonuc.csv")

/n============================================================
B) Zero-shot Mod
/n--- Zero-shot konu özeti ---
 Topic  Count                                          Name
     0     14                     0_motor_güç_altay_türkiye
     1     11                   1_ilk_motor_üretim_muharebe
     2      5 2_sanayii_savunma sanayii_türkiye_den savunma
     3      1                    3_izni_alman_2019_nin 2019
Tamamlandı. Çıktı dosyaları: altay_kesif_sonuc.csv, altay_zeroshot_sonuc.csv
